<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/07.picturing-neural-networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 07: How to Picture Neural Networks - In Your Head and On Paper
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 7 serves as a conceptual synthesis and visual milestone in *Grokking Deep Learning*. Having implemented backpropagation in Chapter 6, Andrew Trask steps back to provide a **10,000-foot view** of neural networks.

Rather than viewing deep learning as a set of equations, Chapter 7 provides mental models to develop intuition for what networks do internally:
1. **Weights as Correlation Filters:** Weights measure alignment between input features and stored patterns.
2. **Hidden Layers as Correlation Creators:** When input features lack direct correlation with targets, hidden layers synthesize intermediate composite features.
3. **Space Warping & Manifolds:** Neural networks fold and warp high-dimensional geometric spaces so non-linear boundaries become linearly separable.

### Key Chapter Takeaways:
- **Correlation Extraction vs. Correlation Creation:** A single-layer network can only evaluate correlations that already exist in the input features. A deep network creates new correlations that make downstream linear separation possible.
- **Visualizing the Geometry:** A neural layer acts as a coordinate transform $\mathbf{z} = \mathbf{x} \mathbf{W}$. The non-linearity (e.g. ReLU) bends the coordinate axes, preventing reverse-collapse.


---
## 2. Theoretical Deep-Dive: The Geometry of Neural Representation

### 2.1 The Two Fundamental Information Flows
In any neural architecture, information flows in two opposing directions:
```
FORWARD PASS: Information / Pattern Flow
[Input Features] ---> [Hidden Representation] ---> [Predictions]
(Evaluates correlations)  (Creates intermediate concepts) (Calculates hypothesis)

BACKWARD PASS: Error / Blame Attribution Flow
[Input Gradient] <--- [Hidden Error Deltas] <--- [Output Error Deltas]
(Propagated via W_01^T) (Gated by activation deriv) (Grad of loss function)
```

### 2.2 Cover's Theorem & Dimensional Expansion
Why does mapping data into a hidden layer help?
**Cover's Theorem on the Separability of Patterns (1965)** states:
> *A complex pattern classification problem, cast non-linearly into a high-dimensional space, is more likely to be linearly separable than in a low-dimensional space.*

In the Streetlights (XOR) problem:
- In the original $2\text{D}$ space, the four points $(0,0), (0,1), (1,0), (1,1)$ cannot be separated by any single straight line.
- Mapping into a $4\text{D}$ or $3\text{D}$ hidden layer via $\mathbf{h} = \text{ReLU}(\mathbf{x} \mathbf{W}_{01})$ lifts the points into higher dimensions where a $2\text{D}$ hyperplane easily slices the classes apart!

### 2.3 The Practical Linear Algebra Connection
In *Practical Linear Algebra* (Mike X Cohen), matrix operations are interpreted geometrically:
- **Matrix Multiplication as Coordinate Transformation:** $\mathbf{y} = \mathbf{A}\mathbf{x}$ stretches, rotates, and shears the space.
- **Rank and Subspaces:** The rank of $\mathbf{W}$ determines the dimensionality of the subspace that can be reached.
- **Kernel & Null Space:** Vectors mapped to zero by $\mathbf{W}$ lie in the null space (features discarded as irrelevant).
- **ReLU as Subspace Partitioning:** ReLU partitions the input space into $2^H$ linear regions (where $H$ is the number of hidden neurons), effectively creating an adaptive piecewise linear manifold.


---
## 3. Code & Thought Experiment: Space Warping Simulation
Let us visualize how a non-linearly separable problem (XOR) is transformed by a hidden layer into linearly separable 3D coordinates.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# XOR / Streetlight points
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
y_xor = np.array([0, 1, 1, 0])

# Pre-trained or representative hidden weights mapping R^2 -> R^3
np.random.seed(42)
W_hidden = np.array([
    [1.5, -1.0, 1.2],
    [-1.0, 1.5, 1.2]
])
b_hidden = np.array([0.0, 0.0, -1.0])

def relu(x):
    return np.maximum(0, x)

# Transform 2D inputs into 3D hidden space
H_space = relu(np.dot(X_xor, W_hidden) + b_hidden)

print("Original 2D coordinates:")
for pt, label in zip(X_xor, y_xor):
    print(f"  Point {pt} -> Label {label}")

print("\nTransformed 3D Hidden Space coordinates:")
for pt, h, label in zip(X_xor, H_space, y_xor):
    print(f"  Point {pt} -> Hidden {np.round(h, 2)} -> Label {label}")

Original 2D coordinates:
  Point [0 0] -> Label 0
  Point [0 1] -> Label 1
  Point [1 0] -> Label 1
  Point [1 1] -> Label 0

Transformed 3D Hidden Space coordinates:
  Point [0 0] -> Hidden [0.  0.  0. ] -> Label 0
  Point [0 1] -> Hidden [0.  1.5 0.2] -> Label 1
  Point [1 0] -> Hidden [1.5 0.  0.2] -> Label 1
  Point [1 1] -> Hidden [0.5 0.5 1.4] -> Label 0

### 3.1 Visualizing the Transformation: 2D Input vs. 3D Hidden Representation
The following plot demonstrates how the hidden layer lifts the entangled points:


In [1]:
fig = plt.figure(figsize=(12, 5))

# 2D Input Space (Entangled XOR)
ax1 = fig.add_subplot(1, 2, 1)
colors = ['crimson' if y == 0 else 'dodgerblue' for y in y_xor]
ax1.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=150, edgecolors='k', zorder=5)
for i, pt in enumerate(X_xor):
    ax1.text(pt[0]+0.05, pt[1]+0.05, f"({pt[0]},{pt[1]}): y={y_xor[i]}", fontsize=10)
ax1.set_xlim(-0.2, 1.4)
ax1.set_ylim(-0.2, 1.4)
ax1.set_title("Input Space $\\mathbb{R}^2$ (Non-Linearly Separable)", fontsize=11)
ax1.set_xlabel("$x_1$")
ax1.set_ylabel("$x_2$")
ax1.grid(True, linestyle=':', alpha=0.6)

# 3D Hidden Space (Lifts Points)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(H_space[:, 0], H_space[:, 1], H_space[:, 2], c=colors, s=150, edgecolors='k', zorder=5)
for i, h in enumerate(H_space):
    ax2.text(h[0]+0.05, h[1]+0.05, h[2]+0.05, f"y={y_xor[i]}", fontsize=9)
ax2.set_title("Hidden Space $\\mathbb{R}^3$ (Linearly Separable via Hyperplane)", fontsize=11)
ax2.set_xlabel("$h_1$")
ax2.set_ylabel("$h_2$")
ax2.set_zlabel("$h_3$")

plt.tight_layout()
plt.show()

<Figure size 1200x500 with 2 Axes>

---
## 4. Correlation Analysis: Input vs. Hidden Representation
Let us compute the empirical correlation matrix between input features and target vs. hidden features and target:


In [1]:
# Correlation of 2D input with label
corr_input = [np.corrcoef(X_xor[:, i], y_xor)[0, 1] for i in range(2)]
print("Correlation of Raw Input Features with Target:")
for i, c in enumerate(corr_input):
    print(f"  Feature x_{i+1}: {c:.4f} (Zero correlation!)")

# Correlation of 3D hidden features with label
corr_hidden = [np.corrcoef(H_space[:, i], y_xor)[0, 1] for i in range(3)]
print("\nCorrelation of Synthesized Hidden Features with Target:")
for i, c in enumerate(corr_hidden):
    print(f"  Hidden Feature h_{i+1}: {c:.4f} (Strong positive/negative correlation created!)")

Correlation of Raw Input Features with Target:
  Feature x_1: 0.0000 (Zero correlation!)
  Feature x_2: 0.0000 (Zero correlation!)

Correlation of Synthesized Hidden Features with Target:
  Hidden Feature h_1: 0.5774 (Strong positive/negative correlation created!)
  Hidden Feature h_2: 0.5774 (Strong positive/negative correlation created!)
  Hidden Feature h_3: -0.7071 (Strong positive/negative correlation created!)

---
## 5. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: What is the primary difference between correlation detection and correlation creation?**  
   *A:* Correlation detection occurs when an output layer calculates $\sum w_i x_i$ against existing input signals. If the input features correlate with the target, this succeeds. Correlation creation occurs when hidden layers combine multiple uncorrelated input features through non-linearities, synthesizing new intermediate signals that directly correlate with the target.

2. **Q: How does Cover's Theorem explain why hidden layers make neural networks capable of learning complex data?**  
   *A:* Cover's theorem demonstrates that projecting entangled, non-linearly separable data into a higher-dimensional space increases the mathematical probability that the classes can be separated by a linear hyperplane.

3. **Q: In linear algebra terms, what is the geometric effect of setting negative activations to zero via ReLU?**  
   *A:* ReLU acts as a projection operator onto the positive orthant $\mathbb{R}^H_+$. It folds the space along coordinate hyperplanes ($z_i = 0$), collapsing negative half-spaces onto the boundary and bending decision regions.
